# Credit Card Default Prediction - Model Experiments

This notebook compares multiple classification models for predicting credit card default risk.

Models tested:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting
5. XGBoost

The goal is to identify the best-performing model before converting the workflow into reusable production scripts.

### Imports and paths ###

In [8]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
from sklearn.model_selection import(
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    RandomizedSearchCV,
    GridSearchCV
)

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
    )

In [15]:
from xgboost import XGBClassifier

In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
)

In [17]:
from sklearn.inspection import permutation_importance

In [18]:
RANDOM_STATE = 999

In [19]:
DATA_PATH = Path("../data/processed/credit_default_cleaned.csv")
REPORT_DIR = Path("../reports")
MODEL_DIR = Path("../models")
IMAGE_DIR = Path("../reports/images")

In [20]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)

### Load cleaned dataset ###

In [21]:
df = pd.read_csv(DATA_PATH)
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (30000, 24)


,credit_limit,sex,education,marriage,age,repayment_status_sep,repayment_status_aug,repayment_status_jul,repayment_status_jun,repayment_status_may,repayment_status_apr,bill_amount_sep,bill_amount_aug,bill_amount_jul,bill_amount_jun,bill_amount_may,bill_amount_apr,payment_amount_sep,payment_amount_aug,payment_amount_jul,payment_amount_jun,payment_amount_may,payment_amount_apr,default_next_month
0,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   credit_limit          30000 non-null  int64
 1   sex                   30000 non-null  int64
 2   education             30000 non-null  int64
 3   marriage              30000 non-null  int64
 4   age                   30000 non-null  int64
 5   repayment_status_sep  30000 non-null  int64
 6   repayment_status_aug  30000 non-null  int64
 7   repayment_status_jul  30000 non-null  int64
 8   repayment_status_jun  30000 non-null  int64
 9   repayment_status_may  30000 non-null  int64
 10  repayment_status_apr  30000 non-null  int64
 11  bill_amount_sep       30000 non-null  int64
 12  bill_amount_aug       30000 non-null  int64
 13  bill_amount_jul       30000 non-null  int64
 14  bill_amount_jun       30000 non-null  int64
 15  bill_amount_may       30000 non-null  int64
 16  bill

In [23]:
df['default_next_month'].value_counts(normalize=True) * 100

default_next_month
0    77.88
1    22.12
Name: proportion, dtype: float64

### Split features and target ###

In [24]:
TARGET = 'default_next_month'
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

print('X shape:', X.shape)
print('y shape:', y.shape)


X shape: (30000, 23)
y shape: (30000,)


### Train-test split ### 

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state = RANDOM_STATE,
    stratify = y
)
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

print('\ny_train distribusion:')
print(y_train.value_counts(normalize=True))

print('\ny_test distribution')
print(y_test.value_counts(normalize=True))

X_train: (22500, 23)
X_test: (7500, 23)

y_train distribusion:
default_next_month
0    0.7788
1    0.2212
Name: proportion, dtype: float64

y_test distribution
default_next_month
0    0.7788
1    0.2212
Name: proportion, dtype: float64


### CV and scoring setup ###

In [27]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state = RANDOM_STATE)
scoring = {'accuracy' : 'accuracy',
           'precision' : 'precision',
           'recall' : 'recall',
           'f1' : 'f1',
           'roc_auc' : 'roc_auc'}

### Baseline models ###

In [30]:
models = {
    'Dummy Classifier': Pipeline(
        steps=[
            ('model', DummyClassifier(strategy='most_frequent'))
        ]
    ),

    'Logistic_Regression': Pipeline(
        steps=[
            ('scalar', StandardScaler()),
            ('model', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced'))
        ]
    ),

    'Decision_Tree': Pipeline(
        steps=[
            ('model', DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'))
        ]
    ),

    'Random_Forest': Pipeline(
        steps=[
            ('model', RandomForestClassifier(n_estimators = 100, random_state = RANDOM_STATE, class_weight='balanced', n_jobs = -1))
        ]
    ),
    
    'Gradient_Boosting': Pipeline(
        steps=[
            ('model', GradientBoostingClassifier(random_state = RANDOM_STATE))
        ]
    ),

    'XGBoost' : Pipeline(
        steps=[
            ('model', XGBClassifier(
                n_estimators = 100,
                random_state = RANDOM_STATE,
                eval_metric = 'logloss',
                n_jobs = -1,
                colsample_bytree = 0.8,
                subsample = 0.8,
                max_depth = 3,
                learning_rate = 0.1
            ))
        ]
    )

}

### Cross-validation function ###

In [32]:
from sklearn.model_selection import cross_validate


cv_scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


def run_cross_validation(model_name, pipeline):
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv = cv,
        scoring = cv_scoring,
        n_jobs = -1,
        return_train_score = True
    )

    return {
        'model': model_name,
        'cv_accuracy': scores['test_accuracy'].mean(),
        'cv_precision': scores['test_precision'].mean(),
        'cv_recall': scores['test_recall'].mean(),
        'cv_f1': scores['test_f1'].mean(),
        'cv_roc_auc': scores['test_roc_auc'].mean()
    }

In [ ]:
from sklearn.model_selection import cross_validate


cv_scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


def run_cross_validation(model_name, pipeline):
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=cv_scoring,
        n_jobs=-1,
        return_train_score=True
    )

    return {
        'model': model_name,
        'cv_accuracy': scores['test_accuracy'].mean(),
        'cv_precision': scores['test_precision'].mean(),
        'cv_recall': scores['test_recall'].mean(),
        'cv_f1': scores['test_f1'].mean(),
        'cv_roc_auc': scores['test_roc_auc'].mean()
    }


cv_results = []
for model_name, pipeline in models.items():
    print(f'Running CV for {model_name}...')
    cv_results.append(run_cross_validation(model_name, pipeline))
cv_results_df = pd.DataFrame(cv_results).sort_values(by='cv_f1', ascending=False)
cv_results_df.sort_values(by='cv_f1', ascending=False, inplace=True)